# Noise Classification

This notebook covers **`swe.noise`**: the ambient-noise detection,
validation, and quality-metric modules. We demonstrate:

1. STA/LTA-based noise window detection with parameter tuning
2. Noise window validation and exploration
3. Statistical analysis and visualization of noise properties

**Dataset**: Synthetic continuous waveforms with embedded noise and signal segments.

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from obspy import Stream, Trace, UTCDateTime

import swe.noise
import swe.viz

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

%matplotlib inline

## Configuration

In [ ]:
SAMPLING_RATE = 100.0
DURATION_HOURS = 1
NPTS = int(SAMPLING_RATE * 3600 * DURATION_HOURS)
NETWORK = "XX"
STATIONS = ["STA1", "STA2", "STA3"]
CHANNEL = "BHZ"
START = UTCDateTime("2024-06-15T00:00:00")
DATA_DIR = Path("data/noise_demo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Generate Synthetic Continuous Data

We create 1-hour traces with:
- Background noise (low amplitude Gaussian)
- Embedded "earthquake" signals (high-amplitude bursts at known times)

In [ ]:
rng = np.random.default_rng(42)
mseed_files = []

for station in STATIONS:
    # Background noise
    data = rng.normal(0, 1.0, NPTS).astype(np.float64)

    # Insert signal bursts at 10, 25, and 45 minutes
    for minute in [10, 25, 45]:
        idx = int(minute * 60 * SAMPLING_RATE)
        burst_len = int(5 * SAMPLING_RATE)  # 5-second burst
        t = np.arange(burst_len) / SAMPLING_RATE
        burst = 50.0 * np.sin(2 * np.pi * 8.0 * t) * np.exp(-0.5 * t)
        data[idx:idx + burst_len] += burst

    tr = Trace(
        data=data,
        header={
            "network": NETWORK,
            "station": station,
            "channel": CHANNEL,
            "sampling_rate": SAMPLING_RATE,
            "starttime": START,
        },
    )
    fname = DATA_DIR / f"{NETWORK}.{station}.{CHANNEL}.{START.year}-{START.month:02d}-{START.day:02d}.ms"
    tr.write(str(fname), format="MSEED")
    mseed_files.append(str(fname))

print(f"Generated {len(mseed_files)} synthetic MiniSEED files")

# Visualize
st = Stream([Trace(data=np.fromfile(f, dtype=np.float64)) for f in mseed_files])
from obspy import read
st = read(mseed_files[0])
fig = swe.viz.plot_stream(st)
fig.set_size_inches(14, 4)
plt.tight_layout()
plt.show()

## 2. STA/LTA Noise Window Detection

The STA/LTA detector compares short-term and long-term averages. Windows where
the ratio stays below a threshold are classified as "noise".

**Parameters**:
- `STA length`: Short-term average window (samples)
- `LTA length`: Long-term average window (samples)
- `Threshold`: STA/LTA ratio below this = noise
- `Window length`: Analysis window size (samples)

In [ ]:
# Default parameters
detection_params = (20, 200, 2.5, 800)  # (sta, lta, threshold, window_length)
time_periods = [(START, START + 3600)]

noise_windows, noise_times = swe.noise.find_noise_windows(
    mseed_files,
    detection_params,
    time_periods,
)

print(f"Noise windows found: {len(noise_times)}")
for i, nt in enumerate(noise_times[:5]):
    print(f"  Window {i}: station={nt[0]}, time={nt[1]}")

## 3. Parameter Sensitivity

This section varies the detection threshold and shows its effect on the number of noise windows found.

In [ ]:
thresholds = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
n_windows = []

for thresh in thresholds:
    params = (20, 200, thresh, 800)
    _, times = swe.noise.find_noise_windows(mseed_files, params, time_periods)
    n_windows.append(len(times))
    print(f"  threshold={thresh:.1f}: {len(times)} windows")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, n_windows, "o-", markersize=8)
ax.set_xlabel("STA/LTA Threshold")
ax.set_ylabel("Number of Noise Windows")
ax.set_title("Noise Detection Sensitivity to Threshold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Noise Window Analysis

Compute statistical metrics for the detected noise windows.

In [ ]:
if len(noise_times) > 0:
    metrics = swe.noise.analyze_noise_windows(noise_windows, noise_times)
    for station, station_metrics in metrics.items():
        print(f"\n{station}: {len(station_metrics)} windows")
        for m in station_metrics[:3]:
            print(f"  max={m[0]:.2f}, median={m[1]:.2f}, rms={m[2]:.2f}")
else:
    print("No noise windows detected - try lowering the threshold.")

## 5. Noise Visualization

In [ ]:
if len(noise_times) > 0:
    fig = swe.noise.plot_noise_time_histogram(
        noise_times, bin_width=300,
        major_x_tick="hour", minor_x_tick="minute",
    )
    plt.show()

    fig = swe.noise.plot_noise_station_histogram(noise_times, STATIONS)
    plt.show()
else:
    print("No noise windows to plot.")

## Summary

| Function | Purpose |
|----------|--------|
| `find_noise_windows()` | STA/LTA-based noise window detection |
| `analyze_noise_windows()` | Statistical metrics (max, median, RMS) |
| `verify_noise_windows()` | Validate detected windows |
| `explore_noise_windows()` | Interactive exploration |
| `plot_noise_time_histogram()` | Temporal distribution |
| `plot_noise_station_histogram()` | Per-station distribution |

Key tuning parameters:
- **STA/LTA lengths**: Scale with sampling rate (STA ~0.2s, LTA ~2s)
- **Threshold**: Lower = more windows, higher = stricter noise criteria
- **Window length**: Controls granularity of noise classification